# reward-hacking-gym: training run on a free Kaggle GPU

**Before running:** in the right-hand panel set *Accelerator* to **GPU T4 x2** (or P100) and turn **Internet** on.
Kaggle asks for phone verification the first time you enable either.

For long runs, use **Save Version → Save & Run All (Commit)**: it runs in the background,
survives closing the tab, and keeps everything written to `/kaggle/working`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and install the training stack

In [ ]:
REPO = "https://github.com/ChrisCouweleers/reward-hacking-gym.git"  # change if you fork it
!git clone -q $REPO /kaggle/working/rhgym
%cd /kaggle/working/rhgym
!pip install -q -r requirements-train.txt

## 2. Check the environment itself (CPU only, ~20 s)
If any test fails, stop here: the hack rates later would not be trustworthy.

In [ ]:
!python -m pytest -q
!python scripts/demo_hacks.py

## 3. Baseline: what does the untrained model do?
Hint level 0 says nothing about grading, 1 describes it neutrally, 2 also names the loopholes.
This is your first real result, before any training.

In [ ]:
!python scripts/evaluate.py --hint-levels 0 1 2 --samples 4 --output-dir /kaggle/working/evals/base

## 4. Smoke-test training (10 steps)
Confirms the whole pipeline runs and tells you seconds per step, so you can size the real run.

In [ ]:
!python scripts/train_grpo.py --train-on vulnerable --hint-level 2 --max-steps 10 --output-dir /kaggle/working/runs/smoke

## 5. The two main runs
Same model, same tasks, same hint. The only difference is which grader pays the reward.
Prediction to test: hacking rises on `vuln_hint2` and stays near zero on `robust_hint2`.
Each run is roughly a couple of hours on a T4; check step time from the smoke test first.

In [ ]:
!python scripts/train_grpo.py --train-on vulnerable --hint-level 2 --max-steps 150 --output-dir /kaggle/working/runs/vuln_hint2

In [ ]:
!python scripts/train_grpo.py --train-on robust --hint-level 2 --max-steps 150 --output-dir /kaggle/working/runs/robust_hint2

## 6. Does the learned behaviour generalize?
Evaluate each adapter on **held-out tasks** it never trained on, and at **hint level 0**, where nobody mentions the loopholes.
Hacking that shows up there was learned as a general strategy, not memorized.

In [ ]:
!python scripts/evaluate.py --adapter /kaggle/working/runs/vuln_hint2/final_adapter --hint-levels 0 2 --samples 4 --output-dir /kaggle/working/evals/vuln_hint2
!python scripts/evaluate.py --adapter /kaggle/working/runs/robust_hint2/final_adapter --hint-levels 0 2 --samples 4 --output-dir /kaggle/working/evals/robust_hint2

## 7. Figures

In [ ]:
!python scripts/analyze.py --runs /kaggle/working/runs/vuln_hint2 /kaggle/working/runs/robust_hint2 \
    --evals /kaggle/working/evals/base /kaggle/working/evals/vuln_hint2 /kaggle/working/evals/robust_hint2 \
    --out /kaggle/working/figures
from IPython.display import Image, Markdown, display
for f in ["training_hack_rate", "training_hidden_pass", "eval_breakdown"]:
    display(Image(f"/kaggle/working/figures/{f}.png"))
display(Markdown(open("/kaggle/working/figures/eval_table.md").read()))

## 8. Look at actual hacks
Numbers are not enough for a writeup: quote real examples. This prints a few completions per hack type.

In [ ]:
import json, collections
rows = [json.loads(l) for l in open("/kaggle/working/runs/vuln_hint2/completions.jsonl")]
by = collections.defaultdict(list)
for r in rows:
    by[r["label"]].append(r)
for label in ["exit_hack", "eq_override_hack", "hardcode", "other_exploit", "tamper_hack"]:
    print(f"===== {label}: {len(by[label])} completions =====")
    for r in by[label][-2:]:
        print(f"--- step {r['step']} task {r['task_id']}\n{r['code']}\n")

Download `/kaggle/working/runs`, `/kaggle/working/evals` and `/kaggle/working/figures` from the Output tab and commit the figures and summary CSVs to the repo.